# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [41]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [42]:
import dask.dataframe as dd


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [43]:
import sys
PRICE_DATA = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))
PRICE_DATA


['../../05_src/data/prices_csv/stocks\\A.csv',
 '../../05_src/data/prices_csv/stocks\\AA.csv',
 '../../05_src/data/prices_csv/stocks\\AACG.csv',
 '../../05_src/data/prices_csv/stocks\\AAL.csv',
 '../../05_src/data/prices_csv/stocks\\AAMC.csv',
 '../../05_src/data/prices_csv/stocks\\AAME.csv',
 '../../05_src/data/prices_csv/stocks\\AAN.csv',
 '../../05_src/data/prices_csv/stocks\\AAOI.csv',
 '../../05_src/data/prices_csv/stocks\\AAON.csv',
 '../../05_src/data/prices_csv/stocks\\AAP.csv',
 '../../05_src/data/prices_csv/stocks\\AAPL.csv',
 '../../05_src/data/prices_csv/stocks\\AAT.csv',
 '../../05_src/data/prices_csv/stocks\\AAU.csv',
 '../../05_src/data/prices_csv/stocks\\AAWW.csv',
 '../../05_src/data/prices_csv/stocks\\AAXN.csv',
 '../../05_src/data/prices_csv/stocks\\AB.csv',
 '../../05_src/data/prices_csv/stocks\\ABB.csv',
 '../../05_src/data/prices_csv/stocks\\ABBV.csv',
 '../../05_src/data/prices_csv/stocks\\ABC.csv',
 '../../05_src/data/prices_csv/stocks\\ABCB.csv',
 '../../05_src

In [44]:
import os
from glob import glob

# Write your code below.

import sys
PRICE_DATA = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
df = dd.read_parquet(parquet_files)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [45]:
# Write your code below.

import dask.dataframe as dd

#read all parquet files
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

#adding lag for variables Close and Adj_Close
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1)))

#adding returns based on Close price
dd_returns = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

#adding hi_lo_range
dd_hi_lo = dd_returns.assign(
    Hi_Lo_Range = lambda x: x['High'] - x['Low']
)

#assigning to dd_feat
dd_feat = dd_hi_lo
dd_feat.head()


C:\Users\Localadmin\AppData\Local\Temp\ipykernel_10244\2971178016.py:9: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns,Hi_Lo_Range
ticker,,,,,,,,,,,,
AAN,2015-01-02,30.809999,30.860001,30.040001,30.620001,30.087009,898900,AAN.csv,2015,NaN,NaN,0.820000
AAN,2015-01-05,30.350000,30.690001,30.230000,30.500000,29.969103,503900,AAN.csv,2015,30.620001,-0.003919,0.460001
AAN,2015-01-06,30.530001,30.540001,29.090000,29.360001,28.848948,1222500,AAN.csv,2015,30.500000,-0.037377,1.450001
AAN,2015-01-07,29.590000,30.350000,29.400000,30.250000,29.723450,917300,AAN.csv,2015,29.360001,0.030313,0.950001
AAN,2015-01-08,30.700001,30.840000,30.420000,30.740000,30.204924,1345300,AAN.csv,2015,30.250000,0.016198,0.420000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [46]:
# Write your code below.

# Converting Dask DataFrame to Pandas DataFrame
df_feat = dd_feat.compute()

# Calculating 10-day moving average of returns
df_feat["returns_ma10"] = df_feat.groupby("ticker")["Returns"].transform(lambda x: x.rolling(10).mean())

df_feat.head(15)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns,Hi_Lo_Range,returns_ma10
ticker,,,,,,,,,,,,,
AAN,1982-11-04,0.0,1.024691,1.012346,1.012346,-1.263439e+21,9182700,AAN.csv,1982,NaN,NaN,0.012346,NaN
AAN,1982-11-05,0.0,1.049383,1.024691,1.024691,-1.278846e+21,2370600,AAN.csv,1982,1.012346,0.012195,0.024691,NaN
AAN,1982-11-08,0.0,1.061728,1.037037,1.037037,-1.294253e+21,1174500,AAN.csv,1982,1.024691,0.012048,0.024691,NaN
AAN,1982-11-09,0.0,1.061728,1.037037,1.037037,-1.294253e+21,1344600,AAN.csv,1982,1.037037,0.000000,0.024691,NaN
AAN,1982-11-10,0.0,1.098765,1.074074,1.074074,-1.340476e+21,819400,AAN.csv,1982,1.037037,0.035714,0.024691,NaN
AAN,1982-11-11,0.0,1.086420,1.061728,1.061728,-1.325069e+21,475200,AAN.csv,1982,1.074074,-0.011494,0.024691,NaN
AAN,1982-11-12,0.0,1.111111,1.086420,1.086420,-1.355885e+21,479200,AAN.csv,1982,1.061728,0.023256,0.024691,NaN
AAN,1982-11-15,0.0,1.086420,1.061728,1.061728,-1.325069e+21,195700,AAN.csv,1982,1.086420,-0.022727,0.024691,NaN
AAN,1982-11-16,0.0,1.061728,1.037037,1.037037,-1.294253e+21,1034100,AAN.csv,1982,1.061728,-0.023256,0.024691,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)


+ **Was it necessary to convert to pandas to calculate the moving average return?**  
No, it wasn't necessary. Dask also supports rolling operations however, rolling operations in Dask can be slower especially if the dataset is small enough to fit into memory.

+ **Would it have been better to do it in Dask? Why?**  
It would have been better to do it in Dask if the data was large and it wouldn't fit into memory. Dask would have distributed the computation and make it scalable.


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.